In [ ]:
%pip install ultralytics opencv-py  thon tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.2 MB/s eta 0:00:00


In [1]:
import sys
print(sys.executable)

c:\AI_Project\Aerial Object Classification and Detection\tf_env\Scripts\python.exe


In [2]:
import os
import cv2
import shutil
from tqdm import tqdm


In [3]:

# Original Dataset Path
train_dir = "C:\\AI_Project\\Aerial Object Classification and Detection\\classification_dataset\\train"
val_dir = "C:\\AI_Project\\Aerial Object Classification and Detection\\classification_dataset\\valid"
test_dir = "C:\\AI_Project\\Aerial Object Classification and Detection\\classification_dataset\\test"

# YOLO dataset folder
output_dir = "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset"
os.makedirs(output_dir, exist_ok=True)

splits = {
    "train": train_dir,
    "val": val_dir,
    "test": test_dir
}

classes = {"bird": 0, "drone": 1}

for split, input_path in splits.items():

    img_out = os.path.join(output_dir, "images", split)
    lbl_out = os.path.join(output_dir, "labels", split)

    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for cls_name, cls_id in classes.items():
        folder = os.path.join(input_path, cls_name)
        images = os.listdir(folder)

        print(f"Processing {split}/{cls_name}...")

        for img_name in tqdm(images):
            img_path = os.path.join(folder, img_name)

            # Loading images
            img = cv2.imread(img_path)
            if img is None:
                continue

            h, w, _ = img.shape

            # Auto-Bounding Box Generation
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            blur = cv2.GaussianBlur(gray, (5, 5), 0)
            _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if len(contours) == 0:
                # fallback: full image bounding box
                x, y, bw, bh = 0, 0, w, h
            else:
                cnt = max(contours, key=cv2.contourArea)
                x, y, bw, bh = cv2.boundingRect(cnt)

            # Converting to YOLO format
            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            width = bw / w
            height = bh / h

            # Save image
            new_img_path = os.path.join(img_out, img_name)
            shutil.copy(img_path, new_img_path)

            # Save label
            label_name = img_name.replace(".jpg", ".txt").replace(".png", ".txt")
            label_path = os.path.join(lbl_out, label_name)

            with open(label_path, "w") as f:
                f.write(f"{cls_id} {x_center} {y_center} {width} {height}\n")


Processing train/bird...


100%|██████████| 1414/1414 [00:10<00:00, 132.35it/s]


Processing train/drone...


100%|██████████| 1248/1248 [00:08<00:00, 140.41it/s]


Processing val/bird...


...

100%|██████████| 225/225 [00:01<00:00, 142.81it/s]


Processing test/bird...


100%|██████████| 121/121 [00:00<00:00, 128.63it/s]


Processing test/drone...


100%|██████████| 94/94 [00:00<00:00, 147.11it/s]


In [4]:
import os
#Checking if the path are right to give in data.yaml
for folder in [
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\images\\train",
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\images\\val",
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\images\\test",
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\labels\\train",
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\labels\\val",
    "C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\labels\\test",
]:
    print(folder, "->", len(os.listdir(folder)))


C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\images\train -> 2662
C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\images\val -> 442
C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\images\test -> 215
C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\labels\train -> 2662
C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\labels\val -> 442
C:\AI_Project\Aerial Object Classification and Detection\detection_dataset\labels\test -> 215


In [2]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
model.train(
    data="data.yaml",
    epochs=70,
    imgsz=640,
    batch=16,
    verbose=True
)


Ultralytics 8.4.37  Python-3.10.0 torch-2.11.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

KeyboardInterrupt: 

- Precision = 0.84
- Recall = 0.84
- maP50 = 0.89
- mAP50-95 = 0.75
- Fitness=0.75

##### My Assumptions
- Strong model for 2-class detection
- Precision and recall are well-balanced, so it’s neither missing objects nor producing too many false positives.


In [1]:
model = YOLO("C:\\AI_Project\\Aerial Object Classification and Detection\\runs\\detect\\train2\\weights\\best.pt")
results = model.predict(
    source="C:\\AI_Project\\Aerial Object Classification and Detection\\detection_dataset\\images\\test\\",
    conf=0.25,
    save=True,
    show=True
)


NameError: name 'YOLO' is not defined

In [ ]:
import shutil
# Zipping the entire runs folder to download
shutil.make_archive("C:\\AI_Project\\Aerial Object Classification and Detection\\train2_runs", 'zip', "C:\\AI_Project\\Aerial Object Classification and Detection\\runs")

'/content/runsdown.zip'